# Using BOW and TF-IDF

In [112]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [113]:
# Load dataset
df_dataset = pd.read_csv('IMDB_Dataset.csv')

# Filter 5000 positive and 5000 negative reviews
df_pos = df_dataset[df_dataset['sentiment'] == 'positive'].head(5000)
df_neg = df_dataset[df_dataset['sentiment'] == 'negative'].head(5000)

# Combine them
df = pd.concat([df_pos, df_neg])

# Reset index
df = df.reset_index(drop=True)

# Check the shape
print(df.shape)

# Display first few rows
df.head()

(10000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,"Petter Mattei's ""Love in the Time of Money"" is...",positive
4,"Probably my all-time favorite movie, a story o...",positive


In [114]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     10000 non-null  object
 1   sentiment  10000 non-null  object
dtypes: object(2)
memory usage: 156.4+ KB


In [115]:
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [116]:
df['sentiment'].value_counts()

sentiment
positive    5000
negative    5000
Name: count, dtype: int64

In [117]:
df.isna().sum()

review       0
sentiment    0
dtype: int64

In [118]:
df.duplicated().sum()

17

In [119]:
df.drop_duplicates(inplace=True)

In [120]:
df.duplicated().sum()

0

In [121]:
# Basic preprocessing
# Remove HTML tags
# lowercase
# remove stopwords

In [122]:
import re
def remove_tags(text):
    tag = re.compile(r'<.*?>')
    return tag.sub(r'', text)

In [123]:
df['review'] = df['review'].apply(remove_tags)
df.sample(5)

,review,sentiment
8413,Eight teen convicts are brought to the abandon...,negative
7383,"Generally speaking, I'm a an admirer of Jess F...",negative
7184,Normally I love finding old (and some not-so-o...,negative
9150,"Alan Rudolph is a so-so director, without that...",negative
7864,"Oh, for crying out loud, this has got to be th...",negative


In [124]:
# lowercase
df['review'] = df['review'].str.lower()
df.sample(5)

,review,sentiment
3755,i saw this flick on the big screen as a kid an...,positive
2677,ernst lubitsch's contribution to the american ...,positive
8866,this documentary is founded on sponge cake as ...,negative
5722,about your terrible movie copying beethoven. a...,negative
4252,i just saw this movie at the tribeca film fest...,positive


In [125]:
# stopwords removal
import nltk
from nltk.corpus import stopwords
stop = stopwords.words('english')
df['review'] = df['review'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop)]))
df.sample(5)

,review,sentiment
8173,"read book book fascinating.this movie, directi...",negative
8924,planet earth suffered terrible environmental d...,negative
5853,watched movie based comments said bad funny. n...,negative
2920,"yet again, early morning television proves inv...",positive
2316,"basis preview seen, went ""shower"" expecting sw...",positive


In [126]:
# Split the data into X and y
X = df['review']
y = df['sentiment']

In [127]:
X

0       one reviewers mentioned watching 1 oz episode ...
1       wonderful little production. filming technique...
2       thought wonderful way spend time hot summer we...
3       petter mattei's "love time money" visually stu...
4       probably all-time favorite movie, story selfle...
                              ...                        
9995    surprisingly plot! ;) seen movies less plot (i...
9996    suppose supposed take something like grain sal...
9997    strictly pryor fans. great, funny guy mean b-m...
9998    saving grace movie serves 0 end movie rating s...
9999    may contain ***spoilers***where start particul...
Name: review, Length: 9983, dtype: object

In [128]:
y

0       positive
1       positive
2       positive
3       positive
4       positive
          ...   
9995    negative
9996    negative
9997    negative
9998    negative
9999    negative
Name: sentiment, Length: 9983, dtype: object

In [129]:
# label encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)
y

array([1, 1, 1, ..., 0, 0, 0])

In [130]:
# split the data into train and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [99]:
X_train.shape

(7986,)

## 1. Using BOW 

In [100]:
# applying BOW
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()
X_train_bow = cv.fit_transform(X_train).toarray()
X_test_bow = cv.transform(X_test).toarray()

In [101]:
X_train_bow.shape

(7986, 48221)

In [102]:
# train naive bayes model
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
gnb.fit(X_train_bow, y_train)

,priors,None
,var_smoothing,1e-09


In [103]:
y_pred = gnb.predict(X_test_bow)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Confusion Matrix: \n", confusion_matrix(y_test, y_pred))
print("Classification Report: \n", classification_report(y_test, y_pred))

Accuracy:  0.614922383575363
Confusion Matrix: 
 [[706 276]
 [493 522]]
Classification Report: 
               precision    recall  f1-score   support

           0       0.59      0.72      0.65       982
           1       0.65      0.51      0.58      1015

    accuracy                           0.61      1997
   macro avg       0.62      0.62      0.61      1997
weighted avg       0.62      0.61      0.61      1997



In [104]:
from sklearn.ensemble import RandomForestClassifier
rfc = RandomForestClassifier()
rfc.fit(X_train_bow, y_train)
y_pred_rfc = rfc.predict(X_test_bow)
print("Accuracy: ", accuracy_score(y_test, y_pred_rfc))
print("Confusion Matrix: \n", confusion_matrix(y_test, y_pred_rfc))
print("Classification Report: \n", classification_report(y_test, y_pred_rfc))

Accuracy:  0.842764146219329
Confusion Matrix: 
 [[837 145]
 [169 846]]
Classification Report: 
               precision    recall  f1-score   support

           0       0.83      0.85      0.84       982
           1       0.85      0.83      0.84      1015

    accuracy                           0.84      1997
   macro avg       0.84      0.84      0.84      1997
weighted avg       0.84      0.84      0.84      1997



## 2. Using TF-IDF

In [105]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray() 
X_train_tfidf.shape

(7986, 48221)

In [106]:
rfc = RandomForestClassifier()
rfc.fit(X_train_tfidf, y_train)
y_pred_rfc_tfidf = rfc.predict(X_test_tfidf)
print("Accuracy: ", accuracy_score(y_test, y_pred_rfc_tfidf))
print("Confusion Matrix: \n", confusion_matrix(y_test, y_pred_rfc_tfidf))
print("Classification Report: \n", classification_report(y_test, y_pred_rfc_tfidf)) 

Accuracy:  0.8417626439659489
Confusion Matrix: 
 [[847 135]
 [181 834]]
Classification Report: 
               precision    recall  f1-score   support

           0       0.82      0.86      0.84       982
           1       0.86      0.82      0.84      1015

    accuracy                           0.84      1997
   macro avg       0.84      0.84      0.84      1997
weighted avg       0.84      0.84      0.84      1997



In [109]:
import xgboost as xgb
xgboost_model = xgb.XGBClassifier()
xgboost_model.fit(X_train_tfidf, y_train)
y_pred_xgb = xgboost_model.predict(X_test_tfidf)
print("Accuracy: ", accuracy_score(y_test, y_pred_xgb))
print("Confusion Matrix: \n", confusion_matrix(y_test, y_pred_xgb))
print("Classification Report: \n", classification_report(y_test, y_pred_xgb))

Accuracy:  0.8217325988983475
Confusion Matrix: 
 [[797 185]
 [171 844]]
Classification Report: 
               precision    recall  f1-score   support

           0       0.82      0.81      0.82       982
           1       0.82      0.83      0.83      1015

    accuracy                           0.82      1997
   macro avg       0.82      0.82      0.82      1997
weighted avg       0.82      0.82      0.82      1997



## 3. Using Word2Vec

In [146]:
import gensim
from gensim.utils import simple_preprocess
from nltk.tokenize import sent_tokenize
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [154]:
# Step 1: Prepare sentences for Word2Vec
story = [
    simple_preprocess(sent)
    for doc in df['review']
    for sent in sent_tokenize(doc)
]

In [155]:
# Step 2: Train Word2Vec model
model = gensim.models.Word2Vec(
    sentences=story,
    vector_size=300,
    window=8,
    min_count=2,
    workers=4,
    epochs=40
)

In [156]:
# Step 3: Function to get average document vectors
def document_vector(doc):
    words = simple_preprocess(doc)
    valid_words = [w for w in words if w in model.wv.index_to_key]
    if not valid_words:
        return np.zeros(model.vector_size)
    return np.mean(model.wv[valid_words], axis=0)

In [157]:
# Step 4: Create feature matrix
X = np.array([document_vector(doc) for doc in tqdm(df['review'].values)])
y = df['sentiment'].map({'positive': 1, 'negative': 0}).values

  0%|          | 0/9983 [00:00<?, ?it/s]

100%|██████████| 9983/9983 [01:50<00:00, 90.04it/s] 


In [158]:
# Step 5: Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Step 6: Train a classifier
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Confusion Matrix: \n", confusion_matrix(y_test, y_pred))
print("Classification Report: \n", classification_report(y_test, y_pred))

Accuracy:  0.8497746619929895
Confusion Matrix: 
 [[838 144]
 [156 859]]
Classification Report: 
               precision    recall  f1-score   support

           0       0.84      0.85      0.85       982
           1       0.86      0.85      0.85      1015

    accuracy                           0.85      1997
   macro avg       0.85      0.85      0.85      1997
weighted avg       0.85      0.85      0.85      1997



In [159]:
rfc = RandomForestClassifier()  
rfc.fit(X_train, y_train)
y_pred = rfc.predict(X_test)
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Confusion Matrix: \n", confusion_matrix(y_test, y_pred))
print("Classification Report: \n", classification_report(y_test, y_pred))

Accuracy:  0.7981972959439159
Confusion Matrix: 
 [[782 200]
 [203 812]]
Classification Report: 
               precision    recall  f1-score   support

           0       0.79      0.80      0.80       982
           1       0.80      0.80      0.80      1015

    accuracy                           0.80      1997
   macro avg       0.80      0.80      0.80      1997
weighted avg       0.80      0.80      0.80      1997



In [160]:
xgboost_model = xgb.XGBClassifier()
xgboost_model.fit(X_train, y_train)
y_pred = xgboost_model.predict(X_test)
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Confusion Matrix: \n", confusion_matrix(y_test, y_pred))
print("Classification Report: \n", classification_report(y_test, y_pred))

Accuracy:  0.8327491236855283
Confusion Matrix: 
 [[814 168]
 [166 849]]
Classification Report: 
               precision    recall  f1-score   support

           0       0.83      0.83      0.83       982
           1       0.83      0.84      0.84      1015

    accuracy                           0.83      1997
   macro avg       0.83      0.83      0.83      1997
weighted avg       0.83      0.83      0.83      1997

